In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, DMatrix, train as xgb_train
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from category_encoders import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [8]:
#Cleaning data 
data = pd.read_csv("2021-2022FootballPlayerStats.csv", sep=";", encoding="ISO-8859-1")
data = data.set_index("Player")

data["Age"] = data["Age"].fillna(data["Age"].mean())

#drop Born, Rk, Squad, Nation
columns_to_drop = ["Born", "Rk", "Squad", "Nation"]

data = data.drop(columns=columns_to_drop)

In [9]:
#Using XGBoost
name_player = "Erling Haaland"

player_with_goals = data.loc[[name_player]]

goals = data["Goals"]
data = data.drop("Goals", axis=1)

index_player = data.index.get_loc(name_player)

categorical_features = ["Pos", "Comp"]
numeric_features = [col for col in data.columns if col not in categorical_features]

data_train, data_test, goals_train, goals_test = train_test_split(
    data, goals, test_size=0.2, random_state=42
)

target_encoder = TargetEncoder(cols=categorical_features)
data_train_enc = target_encoder.fit_transform(data_train, goals_train)
data_test_enc = target_encoder.transform(data_test)

scaler = StandardScaler()
data_train_enc[numeric_features] = scaler.fit_transform(data_train_enc[numeric_features])
data_test_enc[numeric_features] = scaler.transform(data_test_enc[numeric_features])

dtrain = DMatrix(data_train_enc, label=goals_train)
dtest = DMatrix(data_test_enc, label=goals_test)

params = {
    "objective": "reg:squarederror",
    "max_depth": 4,
    "eta": 0.03,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.5,
    "reg_lambda": 1.0,
    "seed": 42
}

bst = xgb_train(
    params,
    dtrain,
    num_boost_round=700,
    evals=[(dtest, "eval")],
    early_stopping_rounds=20,
    verbose_eval=True
)

player_without_goals = data.iloc[[index_player]]
player_enc = target_encoder.transform(player_without_goals)
player_enc[numeric_features] = scaler.transform(player_enc[numeric_features])

dplayer = DMatrix(player_enc)
predicted_goals = bst.predict(dplayer)

print(player_with_goals["Goals"])
print(f"{name_player}: {predicted_goals[0]}")

[0]	eval-rmse:0.19178
[1]	eval-rmse:0.18664
[2]	eval-rmse:0.18191
[3]	eval-rmse:0.17545
[4]	eval-rmse:0.17101
[5]	eval-rmse:0.16622
[6]	eval-rmse:0.16269
[7]	eval-rmse:0.15859
[8]	eval-rmse:0.15552
[9]	eval-rmse:0.15218
[10]	eval-rmse:0.14892
[11]	eval-rmse:0.14642
[12]	eval-rmse:0.14305
[13]	eval-rmse:0.14073
[14]	eval-rmse:0.13765
[15]	eval-rmse:0.13427
[16]	eval-rmse:0.13116
[17]	eval-rmse:0.12846
[18]	eval-rmse:0.12629
[19]	eval-rmse:0.12263
[20]	eval-rmse:0.11886
[21]	eval-rmse:0.11557
[22]	eval-rmse:0.11340
[23]	eval-rmse:0.11243
[24]	eval-rmse:0.11093
[25]	eval-rmse:0.10847
[26]	eval-rmse:0.10667
[27]	eval-rmse:0.10421
[28]	eval-rmse:0.10233
[29]	eval-rmse:0.09984
[30]	eval-rmse:0.09759
[31]	eval-rmse:0.09619
[32]	eval-rmse:0.09479
[33]	eval-rmse:0.09276
[34]	eval-rmse:0.09144
[35]	eval-rmse:0.08982
[36]	eval-rmse:0.08830
[37]	eval-rmse:0.08638
[38]	eval-rmse:0.08445
[39]	eval-rmse:0.08214
[40]	eval-rmse:0.08154
[41]	eval-rmse:0.07968
[42]	eval-rmse:0.07785
[43]	eval-rmse:0.0762

In [11]:
#Error check
y_train_pred = bst.predict(dtrain)
y_test_pred = bst.predict(dtest)

print("Train RMSE:", mean_squared_error(goals_train, y_train_pred))
print("Train MAE:", mean_absolute_error(goals_train, y_train_pred))
print("Train R2:", r2_score(goals_train, y_train_pred))

print("Test RMSE:", mean_squared_error(goals_test, y_test_pred))
print("Test MAE:", mean_absolute_error(goals_test, y_test_pred))
print("Test R2:", r2_score(goals_test, y_test_pred))

Train RMSE: 0.00018632719415227268
Train MAE: 0.005721658515668603
Train R2: 0.9968191511340946
Test RMSE: 0.00047938733756073166
Test MAE: 0.009114081826697004
Test R2: 0.9875501224149188
